In [1]:
!pip install -q langchain langchain-community transformers accelerate torch

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_core.language_models.llms import LLM
from typing import Optional, List

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)


class TinyLlamaLLM(LLM):
    def _call(self, prompt: str, stop=None):
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=128,     
            do_sample=False,         
            temperature=0.0,
        )

        return tokenizer.decode(outputs[0], skip_special_tokens=True)

    @property
    def _llm_type(self) -> str:
        return "tinyllama_custom"


llm = TinyLlamaLLM()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [10]:
from langchain.tools import tool

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def word_length(text: str) -> int:
    """Returns number of characters in text."""
    return len(text)

@tool
def reverse_text(text: str) -> str:
    """Reverses the given text."""
    return text[::-1]

tools = [calculator, word_length, reverse_text]

In [11]:
from langchain_core.prompts import PromptTemplate

template = """Answer the following question using the tools provided.

You have access to the following tools:
{tools}

Tool names: {tool_names}

Use this format:

Question: {input}
Thought: think about what to do
Action: one of [{tool_names}]
Action Input: input to the tool
Observation: result of the tool
... (repeat as needed)
Thought: I now know the final answer
Final Answer: the answer

{agent_scratchpad}
"""

prompt = PromptTemplate.from_template(template)

In [12]:
from langchain_classic.agents import create_react_agent, AgentExecutor

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=3,         
    early_stopping_method="force"
)

In [13]:
import torch
torch.cuda.empty_cache()

In [14]:
response = agent_executor.invoke({"input": "What is 15 * 6?"})
print(response)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




> Entering new AgentExecutor chain...


Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Parsing LLM output produced both a final answer and a parse-able action:: Answer the following question using the tools provided.

You have access to the following tools:
calculator(expression: str) -> str - Evaluate a mathematical expression.
word_length(text: str) -> int - Returns number of characters in text.
reverse_text(text: str) -> str - Reverses the given text.

Tool names: calculator, word_length, reverse_text

Use this format:

Question: What is 15 * 6?
Thought: think about what to do
Action: one of [calculator, word_length, reverse_text]
Action Input: input to the tool
Observation: result of the tool
... (repeat as needed)
Thought: I now know the final answer
Final Answer: the answer


Question: What is the length of the longest word in the given text?
Thought: think about what to do
Action: one of [calculator, word_length, reverse_text]
Action Input: input to the tool
Observation: result of the tool
... (repeat as needed)
Thought: I now know the length of the longest word
F

Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Parsing LLM output produced both a final answer and a parse-able action:: Answer the following question using the tools provided.

You have access to the following tools:
calculator(expression: str) -> str - Evaluate a mathematical expression.
word_length(text: str) -> int - Returns number of characters in text.
reverse_text(text: str) -> str - Reverses the given text.

Tool names: calculator, word_length, reverse_text

Use this format:

Question: What is 15 * 6?
Thought: think about what to do
Action: one of [calculator, word_length, reverse_text]
Action Input: input to the tool
Observation: result of the tool
... (repeat as needed)
Thought: I now know the final answer
Final Answer: the answer

Parsing LLM output produced both a final answer and a parse-able action:: Answer the following question using the tools provided.

You have access to the following tools:
calculator(expression: str) -> str - Evaluate a mathematical expression.
word_length(text: str) -> int - Returns number of c